In [3]:
import sys
sys.path.insert(0, "../../src")

import numpy as np
import jax
import jax.numpy as jnp
from scipy.optimize import least_squares
from temgym_core.transfer_matrices import propagation_matrix, lens_matrix

jax.config.update("jax_enable_x64", True)

# ---------------------------
# Physics helpers (JAX-safe)
# ---------------------------
E_CHARGE = 1.602176634e-19
M_E      = 9.1093837015e-31
C_LIGHT  = 299792458.0
MU_0     = 1.25663706212e-6

def K_rot(U_kV, xp=jnp):
    U = U_kV * 1e3
    gamma = 1.0 + E_CHARGE * U / (M_E * C_LIGHT**2)
    v = C_LIGHT * xp.sqrt(1.0 - 1.0 / gamma**2)
    return E_CHARGE * MU_0 / (2.0 * M_E * v)

def eta_sq(U_kV, U_ref_kV=200.0, xp=jnp):
    # Current implementation: eta^2 = (v/v_ref)^2
    return (K_rot(U_ref_kV, xp=xp) / K_rot(U_kV, xp=xp))**2  # since K_rot ~ 1/v

# ---------------------------
# Core forward model
# ---------------------------
def build_abcd(dists, focals, xp=jnp):
    M = propagation_matrix(dists[-1], xp=xp)
    for i in reversed(range(len(focals))):
        M = M @ lens_matrix(focals[i], xp=xp)
        M = M @ propagation_matrix(dists[i], xp=xp)
    return M

def f_from_excitation(Cf, I0, w, eta_sq_val):
    return 1.0 / (Cf * eta_sq_val * (I0 * (1.0 + w))**2)

def psi_from_excitation(Kv, I0, w):
    return Kv * I0 * (1.0 + w)

def model_measurement(dists, I0, Cf, Kv, wobble_lens, w, defocus, eta_sq_val):
    # distances with defocus applied to first drift
    d = jnp.array(dists)
    d = d.at[0].add(defocus)

    # focal lengths: only the selected lens sees wobble
    idx = jnp.arange(I0.shape[0])
    w_vec = jnp.where(idx == wobble_lens, w, 0.0)
    f = f_from_excitation(Cf, I0, w_vec, eta_sq_val)

    M = build_abcd(d, f, xp=jnp)
    A, B = M[0, 0], M[0, 1]

    psi = jnp.sum(psi_from_excitation(Kv, I0, w_vec))
    return A, B, psi

# ---------------------------
# Measurement generation
# ---------------------------
def generate_measurements(
    d_true, I0_true, Cf_true,
    wobble_values, defocus_values,
    voltages_kV=(200.0,),
    include_B=True,
):
    """
    Returns dict of NumPy arrays.
    For multi-voltage, each sample stores precomputed Kv and eta_sq_val.
    """
    rows = []
    for U in voltages_kV:
        Kv = float(K_rot(U, xp=np))
        esq = float(eta_sq(U, xp=np))
        for wl in range(len(I0_true)):
            for w in wobble_values:
                for df in defocus_values:
                    A, B, psi = model_measurement(
                        d_true, jnp.array(I0_true), jnp.array(Cf_true),
                        Kv, wl, float(w), float(df), esq
                    )
                    rows.append((U, wl, w, df, float(A), float(B), float(psi), esq, Kv))

    arr = np.array(rows, dtype=float)
    out = {
        "voltage_kV":   arr[:, 0],
        "wobble_lens":  arr[:, 1].astype(np.int32),
        "wobble":       arr[:, 2],
        "defocus":      arr[:, 3],
        "A":            arr[:, 4],
        "psi":          arr[:, 6],
        "eta_squared":  arr[:, 7],
        "Kv":           arr[:, 8],
    }
    if include_B:
        out["B"] = arr[:, 5]
    return out

def add_relative_noise(meas, noise_level=0.0, seed=0, keys=("A","B","psi")):
    if noise_level <= 0:
        return meas
    rng = np.random.default_rng(seed)
    meas = dict(meas)  # shallow copy
    for k in keys:
        if k in meas:
            x = np.asarray(meas[k])
            meas[k] = x + rng.normal(0.0, noise_level * np.abs(x), size=x.shape)
    return meas

# ---------------------------
# Residual builder
# ---------------------------
def make_residual_fn(
    meas, n_lenses,
    *,
    use_B: bool,
    total_distance: float | None,
    scales=(1.0, 1.0, 1.0),
    lambda_tikh=0.0,
    lambda_smooth_cf=0.0,
    lambda_smooth_d=0.0,
):
    wl  = jnp.array(meas["wobble_lens"])
    w   = jnp.array(meas["wobble"])
    df  = jnp.array(meas["defocus"])
    A_m = jnp.array(meas["A"])
    psi_m = jnp.array(meas["psi"])
    esq = jnp.array(meas["eta_squared"])
    Kv  = jnp.array(meas["Kv"])
    if use_B:
        B_m = jnp.array(meas["B"])

    A_scale = scales[0]
    B_scale = scales[1] if use_B else 1.0
    psi_scale = scales[2] if use_B else scales[1]

    n_dist_fit = n_lenses if total_distance is not None else (n_lenses + 1)
    n_I0 = n_lenses

    @jax.jit
    def rfn_jit(params):
        # unpack
        if total_distance is not None:
            d_fit = params[:n_dist_fit]
            d_last = total_distance - jnp.sum(d_fit)
            d = jnp.concatenate([d_fit, jnp.array([d_last])])
        else:
            d = params[:n_dist_fit]

        I0 = params[n_dist_fit : n_dist_fit + n_I0]
        Cf = params[n_dist_fit + n_I0 : n_dist_fit + 2*n_I0]

        def one(wl_i, w_i, df_i, esq_i, Kv_i):
            return model_measurement(d, I0, Cf, Kv_i, wl_i, w_i, df_i, esq_i)

        A_p, B_p, psi_p = jax.vmap(one)(wl, w, df, esq, Kv)

        res = [ (A_p - A_m)/A_scale, (psi_p - psi_m)/psi_scale ]
        if use_B:
            res.insert(1, (B_p - B_m)/B_scale)  # A, B, psi

        # regularization
        reg = []
        if lambda_tikh > 0:
            reg.append(jnp.sqrt(lambda_tikh) * params)
        if lambda_smooth_cf > 0:
            reg.append(jnp.sqrt(lambda_smooth_cf) * jnp.diff(Cf))
        if lambda_smooth_d > 0:
            reg.append(jnp.sqrt(lambda_smooth_d) * jnp.diff(d))

        if reg:
            res.extend(reg)
        return jnp.concatenate(res)

    # SciPy wants NumPy
    def rfn_np(x):
        return np.asarray(rfn_jit(jnp.array(x)))

    return rfn_np

# ---------------------------
# Multi-start optimizer
# ---------------------------
def solve_least_squares(rfn, x0, bounds, n_starts=20, refine=10, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    best_x = None
    best_loss = np.inf

    lo, hi = bounds

    for s in range(n_starts):
        scale = 0.8 + 0.4*rng.random(x0.shape[0])
        x = np.clip(x0 * scale, lo + 1e-12, hi - 1e-12)

        sol = least_squares(rfn, x, bounds=bounds, method="trf",
                            ftol=1e-8, xtol=1e-8, gtol=1e-8, max_nfev=7000)
        loss = np.sum(rfn(sol.x)**2)
        if loss < best_loss:
            best_loss = loss
            best_x = sol.x
        if verbose and (s+1) % max(1, n_starts//5) == 0:
            print(f"  start {s+1}/{n_starts}: best loss {best_loss:.3e}")

    for _ in range(refine):
        sol = least_squares(rfn, best_x, bounds=bounds, method="trf",
                            ftol=1e-11, xtol=1e-11, gtol=1e-11, max_nfev=20000)
        loss = np.sum(rfn(sol.x)**2)
        if loss < best_loss:
            best_loss = loss
            best_x = sol.x

    return best_x, best_loss


In [ ]:
# ================================================================
# CONFIGURATION CELL
# ================================================================
import numpy as np

# -------------------------------
# Experiment / problem definition
# -------------------------------
N_LENSES = 5

# Measurement modes:
#   "single"                      -> 200 kV only, single defocus plane
#   "single_defocus_sweep"        -> 200 kV only, defocus sweep
#   "multi_no_B"                  -> multiple voltages, NO B
#   "multi_voltage_with_defocus"  -> multiple voltages, WITH B
MEASUREMENT_MODE = "single_defocus_sweep"

# Noise (relative, i.e. sigma = level * |signal|)
NOISE_LEVEL = 0.01
DISTANCE_MEASUREMENT_ERROR = 0.01  # applied as extra relative noise to A/B (if present)

# Optimizer
N_STARTS = 20
REFINE_RUNS = 20
VERBOSE = True

# Regularization (set to 0 automatically if NOISE_LEVEL == 0)
LAMBDA_TIKHONOV      = 0.0 if NOISE_LEVEL == 0 else 0
LAMBDA_SMOOTH_CF     = 0.0 if NOISE_LEVEL == 0 else 0
LAMBDA_SMOOTH_DIST   = 0.0

# Use the total distance constraint (recommended; matches your old notebook)
USE_TOTAL_DISTANCE_CONSTRAINT = True

# -------------------------------
# Wobble / defocus grids
# -------------------------------
WOBBLE_CONFIG = {2: 3, 3: 3, 4: 3, 5: 5}
DEFOCUS_CONFIG = {2: 3, 3: 3, 4: 3, 5: 5}

n_wobbles = WOBBLE_CONFIG.get(N_LENSES, 5)
n_defocus = DEFOCUS_CONFIG.get(N_LENSES, 5)

wobble_range = np.linspace(-0.1, 0.1, n_wobbles)
DEFOCUS_RANGE = np.linspace(-50e-3, 50e-3, n_defocus)

# Voltages for multi-voltage modes
VOLTAGES_KV = [200.0]

# -------------------------------
# Define test systems
# -------------------------------
SYSTEMS = {
    2: {"D": np.array([0.010, 0.008, 0.012]),
        "I0": np.array([1000.0, 1200.0]),
        "Cf": np.array([1e-5, 1.2e-5])},
    3: {"D": np.array([0.010, 0.008, 0.012, 0.015]),
        "I0": np.array([1000.0, 1200.0, 900.0]),
        "Cf": np.array([1e-5, 1.2e-5, 0.9e-5])},
    4: {"D": np.array([0.010, 0.008, 0.012, 0.015, 0.009]),
        "I0": np.array([1000.0, 1200.0, 900.0, 1100.0]),
        "Cf": np.array([1e-5, 1.2e-5, 0.9e-5, 1.1e-5])},
    5: {"D": np.array([0.010, 0.008, 0.012, 0.015, 0.009, 0.011]),
        "I0": np.array([1000.0, 1200.0, 900.0, 1100.0, 950.0]),
        "Cf": np.array([1e-5, 1.2e-5, 0.9e-5, 1.1e-5, 0.95e-5])},
}

sys_params = SYSTEMS[N_LENSES]
d_true = sys_params["D"]
i0_true = sys_params["I0"]
cf_true = sys_params["Cf"]

total_distance = float(d_true.sum()) if USE_TOTAL_DISTANCE_CONSTRAINT else None

# -------------------------------
# Select measurement plan
# -------------------------------
if MEASUREMENT_MODE == "single":
    voltages = (200.0,)
    defocus_vals = np.array([0.0])
    include_B = True

elif MEASUREMENT_MODE == "single_defocus_sweep":
    voltages = (200.0,)
    defocus_vals = DEFOCUS_RANGE
    include_B = True

elif MEASUREMENT_MODE == "multi_no_B":
    voltages = tuple(VOLTAGES_KV)
    defocus_vals = DEFOCUS_RANGE
    include_B = False

elif MEASUREMENT_MODE == "multi_voltage_with_defocus":
    voltages = tuple(VOLTAGES_KV)
    defocus_vals = DEFOCUS_RANGE
    include_B = True

else:
    raise ValueError(f"Unknown MEASUREMENT_MODE: {MEASUREMENT_MODE}")

wobble_vals = wobble_range

print(f"\n{'='*70}")
print(f"CONFIG: N={N_LENSES}")
print(f"  Mode: {MEASUREMENT_MODE}")
print(f"  Voltages (kV): {list(voltages)}")
print(f"  Wobbles: {len(wobble_vals)} from {wobble_vals[0]:.3f} to {wobble_vals[-1]:.3f}")
print(f"  Defocus: {len(defocus_vals)} planes from {defocus_vals[0]*1e3:.1f} to {defocus_vals[-1]*1e3:.1f} mm")
print(f"  Include B: {include_B}")
print(f"  Noise: {NOISE_LEVEL*100:.1f}% | Distance-error-as-noise: {DISTANCE_MEASUREMENT_ERROR*100:.1f}%")
print(f"  Total-distance constraint: {total_distance is not None} (total={total_distance:.6f} m)")
print(f"  Reg: tikh={LAMBDA_TIKHONOV:.0e}, smoothCf={LAMBDA_SMOOTH_CF:.0e}, smoothD={LAMBDA_SMOOTH_DIST:.0e}")
print(f"{'='*70}\n")

# -------------------------------
# Generate measurements
# -------------------------------
meas = generate_measurements(
    d_true, i0_true, cf_true,
    wobble_values=wobble_vals,
    defocus_values=defocus_vals,
    voltages_kV=voltages,
    include_B=include_B
)

# Inject noise safely (only touches keys that exist)
meas = add_relative_noise(meas, NOISE_LEVEL, seed=42, keys=("A", "B", "psi"))

# Add “distance measurement error” as additional relative noise to A/B only
if DISTANCE_MEASUREMENT_ERROR > 0:
    meas = add_relative_noise(meas, DISTANCE_MEASUREMENT_ERROR, seed=777, keys=("A", "B"))

print(f"Generated {len(meas['A'])} measurements")
print(f"  A:   min={np.min(meas['A']):.3e}, max={np.max(meas['A']):.3e}, std={np.std(meas['A']):.3e}")
if "B" in meas:
    print(f"  B:   min={np.min(meas['B']):.3e}, max={np.max(meas['B']):.3e}, std={np.std(meas['B']):.3e}")
print(f"  psi: min={np.min(meas['psi']):.3e}, max={np.max(meas['psi']):.3e}, std={np.std(meas['psi']):.3e}")

# -------------------------------
# Build residual function
# -------------------------------
if include_B:
    scales = (1.0, 1.0, 1.0)  # A, B, psi
else:
    scales = (1.0, 1.0)       # A, psi

rfn = make_residual_fn(
    meas, N_LENSES,
    use_B=include_B,
    total_distance=total_distance,
    scales=scales,
    lambda_tikh=LAMBDA_TIKHONOV,
    lambda_smooth_cf=LAMBDA_SMOOTH_CF,
    lambda_smooth_d=LAMBDA_SMOOTH_DIST,
)

# -------------------------------
# Parameter bounds and initial guess
# -------------------------------
# Parameter vector convention:
#   if total_distance is not None: [d1..dN, I0_1..I0_N, Cf_1..Cf_N]  length 3N
#   else:                         [d1..d_{N+1}, I0_1..I0_N, Cf_1..Cf_N] length 3N+1
n_dist_fit = N_LENSES if total_distance is not None else (N_LENSES + 1)
n_params = n_dist_fit + 2 * N_LENSES

lower = np.concatenate([
    np.full(n_dist_fit, 1e-3),    # distances
    np.full(N_LENSES, 10.0),      # currents
    np.full(N_LENSES, 5e-7),      # Cf
])
upper = np.concatenate([
    np.full(n_dist_fit, 0.5),
    np.full(N_LENSES, 30000.0),
    np.full(N_LENSES, 5e-4),
])

rng = np.random.default_rng(123)
if total_distance is not None:
    d_init = total_distance * (rng.random(N_LENSES) + 0.1)
    d_init = d_init / d_init.sum() * total_distance
else:
    d_init = total_distance * (rng.random(N_LENSES + 1) + 0.1)
    d_init = d_init / d_init.sum() * total_distance

i0_init = rng.uniform(100, 5000, N_LENSES)
cf_init = 10.0 ** rng.uniform(-6, -4, N_LENSES)
x0 = np.concatenate([d_init, i0_init, cf_init])

print(f"\nFitting {n_params} parameters with {len(meas['A'])} measurements...")

# -------------------------------
# Run optimizer
# -------------------------------
best_x, best_loss = solve_least_squares(
    rfn, x0, bounds=(lower, upper),
    n_starts=N_STARTS, refine=REFINE_RUNS,
    seed=0, verbose=VERBOSE
)

print(f"\nFinal loss: {best_loss:.6e}")

# -------------------------------
# Report results (like old notebook)
# -------------------------------
x_true = np.concatenate([d_true, i0_true, cf_true])

if total_distance is not None:
    d_fit = best_x[:N_LENSES]
    d_last = total_distance - np.sum(d_fit)
    x_fit = np.concatenate([d_fit, np.array([d_last]), best_x[N_LENSES:]])
else:
    # Here distances are all explicitly in the fit
    x_fit = np.concatenate([best_x[:N_LENSES+1], best_x[N_LENSES+1:]])

errors = 100.0 * np.abs((x_fit - x_true) / (x_true + 1e-30))
print(f"Max error: {np.max(errors):.3f}%")

print("\nDistances (m):")
for i in range(N_LENSES + 1):
    print(f"  d[{i}]: true={x_true[i]:.6e}  fit={x_fit[i]:.6e}  err={errors[i]:.3f}%")

print("\nCurrents I0 (A):")
offset = (N_LENSES + 1)
for i in range(N_LENSES):
    idx = offset + i
    print(f"  I0[{i}]: true={x_true[idx]:.6e}  fit={x_fit[idx]:.6e}  err={errors[idx]:.3f}%")

print("\nFocal coefficients Cf:")
offset = (N_LENSES + 1 + N_LENSES)
for i in range(N_LENSES):
    idx = offset + i
    print(f"  Cf[{i}]: true={x_true[idx]:.6e}  fit={x_fit[idx]:.6e}  err={errors[idx]:.3f}%")



CONFIG: N=5
  Mode: single_defocus_sweep
  Voltages (kV): [200.0]
  Wobbles: 5 from -0.100 to 0.100
  Defocus: 5 planes from -50.0 to 50.0 mm
  Include B: True
  Noise: 1.0% | Distance-error-as-noise: 1.0%
  Total-distance constraint: True (total=0.065000 m)
  Reg: tikh=0e+00, smoothCf=0e+00, smoothD=0e+00

Generated 125 measurements
  A:   min=-6.005e-01, max=-3.727e-01, std=4.180e-02
  B:   min=-3.738e-03, max=5.578e-02, std=1.711e-02
  psi: min=2.630e+00, max=2.842e+00, std=4.783e-02

Fitting 15 parameters with 125 measurements...
  start 4/20: best loss 8.525e-02
  start 8/20: best loss 8.525e-02
  start 12/20: best loss 8.525e-02
  start 16/20: best loss 8.525e-02
  start 20/20: best loss 8.525e-02

Final loss: 8.524827e-02
Max error: 49.885%

Distances (m):
  d[0]: true=1.000000e-02  fit=1.038078e-02  err=3.808%
  d[1]: true=8.000000e-03  fit=7.967266e-03  err=0.409%
  d[2]: true=1.200000e-02  fit=1.382136e-02  err=15.178%
  d[3]: true=1.500000e-02  fit=1.035447e-02  err=30.970%

# Mathematical Foundation: Lens Inversion Model

## Electromagnetic Lens Theory (Glaser Bell Model)

### Focal Length

The focal length of an electromagnetic (solenoid) lens follows the **Glaser bell model**:

$$f_i = \frac{1}{C_{f,i} \cdot \eta(U)^2 \cdot [I_{0,i}(1+w_i)]^2}$$

**Parameters:**
- $C_{f,i}$ = focal length coefficient for lens $i$ (units: m⁻¹) — **device geometry constant**
- $I_{0,i}$ = magnetic excitation (coil current, units: A) — **what we fit**
- $w_i$ = fractional current variation (wobble, dimensionless) — **measurement probe**
- $\eta(U)^2$ = voltage scaling factor (dimensionless) — depends on accelerating voltage

### Voltage Scaling Factor

$$\eta(U) = \frac{v(U)}{v_{\text{ref}}}$$

where $v(U)$ is the electron velocity at accelerating voltage $U$ relative to reference velocity at 200 kV.

$$v(U) = c \sqrt{1 - \frac{1}{\gamma(U)^2}}, \quad \gamma(U) = 1 + \frac{eU}{m_e c^2}$$

**Constants used:**
- $e = 1.602 \times 10^{-19}$ C (elementary charge)
- $m_e = 9.109 \times 10^{-31}$ kg (electron rest mass)
- $c = 2.998 \times 10^8$ m/s (speed of light)
- At 200 kV: $\eta(200 \text{ kV})^2 = 1.0$ (normalization reference)
- At 180 kV: $\eta(180 \text{ kV})^2 \approx 0.937$ (~6% change)
- At 220 kV: $\eta(220 \text{ kV})^2 \approx 1.058$ (~6% change)

### Rotation (Deflection by Magnetic Field)

$$\psi_i = R(U) \cdot I_{0,i}(1+w_i)$$

where the rotation constant is:

$$K_v(U) = \frac{e\mu_0}{2m_e v(U)}$$

**Constants:**
- $\mu_0 = 1.257 \times 10^{-6}$ H/m (permeability of free space)
- At 200 kV: $K_v(200 \text{ kV}) \approx 5.3 \times 10^{-4}$ rad/A
- Varies ~1.5% per 10 kV (inverse relationship with velocity)

**Total rotation:** $\psi_{\text{total}} = \sum_{i=1}^N \psi_i$ (sum over all lenses)

## ABCD Matrix Formalism

The lens system is described by the **paraxial ray equation** using 2×2 transfer matrices:

$$\begin{pmatrix} y_{\text{out}} \\ y'_{\text{out}} \end{pmatrix} = M \begin{pmatrix} y_{\text{in}} \\ y'_{\text{in}} \end{pmatrix}$$

where $y$ is position and $y'$ is angle.

**Propagation matrix** (field-free drift distance $d$):
$$M_{\text{drift}}(d) = \begin{pmatrix} 1 & d \\ 0 & 1 \end{pmatrix}$$

**Lens matrix** (focal length $f$):
$$M_{\text{lens}}(f) = \begin{pmatrix} 1 & 0 \\ -1/f & 1 \end{pmatrix}$$

**System matrix** (composed in reverse order):
$$M_{\text{system}} = M_{\text{drift}, N} \cdot M_{\text{lens}, N} \cdots M_{\text{lens}, 1} \cdot M_{\text{drift}, 1}$$

Output parameters:
- $A = M_{00}$ (magnification for unit input position)
- $B = M_{01}$ (input-output position coupling)

## Measurement Model

For each measurement configuration (voltage, wobble, defocus):

1. **Compute voltage-dependent factor:** $\eta^2(U)$ and $K_v(U)$
2. **Apply wobble:** Only lens $\ell$ receives current variation
3. **Apply defocus:** Add offset $\Delta d$ to first drift distance: $d_1' = d_1 + \Delta d$
4. **Forward model:**
   - Focal lengths: $f_i = \frac{1}{C_{f,i} \eta^2 [I_{0,i}(1+w_i)]^2}$
   - Build ABCD matrix $M$ from distances and focal lengths
   - Extract: $A = M_{00}$, $B = M_{01}$
   - Rotation: $\psi = K_v \sum_i I_{0,i}(1+w_i)$

## Inverse Problem & Why It's Hard

**Measurement-to-parameter relationship:**

$$A, B, \psi \approx f(d_{\text{all}}, I_{0,\text{all}}, C_{f,\text{all}}, U)$$

**Challenges:**
1. **I₀/Cf degeneracy:** Both increase focal length similarly — need B measurement to break this
2. **Distance ambiguity:** Small focal length ~ large distance OR small Cf — need systematic variation to separate
3. **Nonlinearity:** ABCD matrix multiplication creates complex coupling

**Solution: Multi-measurement strategy**
- **Wobble variation** → Isolates individual lens effects
- **Defocus sweep** → Couples distance measurements through ABCD matrix structure  
- **B measurement** → Independent constraint on focal lengths (captures $1/f$ directly)
- **Multi-voltage** → Adds voltage-dependent leverage on rotation ($K_v(U)$ variation)
- **Regularization** → Penalizes unphysical solutions in presence of noise